# Etapa 5. Verificação da produtividade potencial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/05_verificacao_pp.ipynb)

Antes de calibrar o ky, é preciso saber se a **PP está numa faixa realista**. A PP é o teto: com manejo perfeito
e sem falta de água. Nenhum município deveria colher acima dela de forma sistemática.

- PP **baixa demais** esconde o yield gap de manejo (o produtor parece já colher "tudo");
- PP **alta demais** infla o yield gap de manejo.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# No Colab, esta célula:
#   1) instala as bibliotecas que o Colab não tem;
#   2) copia o projeto do GitHub (dados + funções) para a máquina do Colab;
#   3) conecta o seu Google Drive, onde os resultados ficam guardados entre uma sessão e outra.
# Fora do Colab (no computador do orientador), ela só acerta as pastas.
import os
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if NO_COLAB:
    !pip -q install agrometeorologiapy==1.0.1
    if Path("/content/milho_oeste").exists():
        !git -C /content/milho_oeste pull -q     # já copiado nesta sessão: só atualiza
    else:
        !git clone -q https://github.com/fcoliveira-utfpr/milho_oeste.git /content/milho_oeste
    os.chdir("/content/milho_oeste")             # entra na pasta do projeto
    from google.colab import drive
    drive.mount("/content/drive")                # pede autorização na primeira vez
    SAIDA = Path("/content/drive/MyDrive/milho_oeste/resultados")
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    SAIDA = Path("resultados")

SAIDA.mkdir(parents=True, exist_ok=True)         # cria a pasta de resultados, se não existir
sys.path.append("src")                           # permite importar as funções do projeto (pasta src/)
print("Pasta do projeto:", Path.cwd())
print("Resultados em   :", SAIDA)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mza_oeste as M

sim = pd.read_parquet(SAIDA / "simulacao_mza_oeste.parquet")
ibge = pd.read_csv("dados/ibge_pam_milho_oeste.csv")
ibge = ibge[ibge["safra"] == 2]

jan = sim[sim["semeadura"].isin(M.JANELA)].groupby(["cod_ibge", "ano"])[["PPf", "PA"]].mean().reset_index()
cmp = jan.merge(ibge[["cod_ibge", "ano", "rendimento_kg_ha"]], on=["cod_ibge", "ano"])
print(len(cmp), "município-anos com simulação e IBGE")

## 1. Quantas vezes o IBGE passa da PP?

In [ ]:
cmp["PR_sobre_PP"] = cmp["rendimento_kg_ha"] / cmp["PPf"]
print(f"Município-anos com PR > PP: {(cmp['PR_sobre_PP'] > 1).mean():.1%}")
print(f"Nos últimos 5 anos (2021-2025): {(cmp.loc[cmp['ano'] >= 2021, 'PR_sobre_PP'] > 1).mean():.1%}")
cmp.groupby("ano")["PR_sobre_PP"].describe()[["mean", "50%", "max"]].round(2)

In [ ]:
anual = cmp.groupby("ano").agg(PP=("PPf", "mean"), PR_p95=("rendimento_kg_ha", lambda x: x.quantile(0.95)),
                               PR_media=("rendimento_kg_ha", "mean"))
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(anual.index, anual["PP"], marker="o", label="PP simulada (média da janela)")
ax.plot(anual.index, anual["PR_p95"], marker="s", label="IBGE, percentil 95")
ax.plot(anual.index, anual["PR_media"], marker="^", label="IBGE, média")
ax.set_ylabel("kg/ha")
ax.legend()
fig.savefig(SAIDA / "fig_pp_vs_ibge.png", dpi=300, bbox_inches="tight")
plt.show()

## 2. Outras referências de alto rendimento

Complete à mão a tabela abaixo antes de decidir:

| Fonte | Produtividade (kg/ha) | Observação |
|---|---|---|
| Ensaios estaduais de cultivares do IDR-Paraná, milho safrinha, locais do Oeste (média dos 5 melhores híbridos) | [preencher] | buscar no site do IDR-Paraná |
| GYGA, milho de sequeiro, zona climática do Oeste do PR: Yp e Yw | [preencher] | www.yieldgap.org |
| Maior produtividade do IBGE na série (município, ano) | ver abaixo | |

In [ ]:
cmp.sort_values("rendimento_kg_ha", ascending=False).head(5).merge(
    sim[["cod_ibge", "municipio"]].drop_duplicates(), on="cod_ibge")

## 3. Decisão: ajustar a PP?

No MZA-FAO a PP é **proporcional ao índice de colheita (Cc)**: PP = (produção de matéria seca) × Cc / (1 − U).
O projeto usa Cc = 0,40, valor da época de Doorenbos & Kassam. Híbridos modernos têm índice de colheita perto de
0,50. Multiplicar a PP por um **fator** equivale a usar outro Cc:

$$Cc_{ajustado} = 0{,}40 \times fator$$

Uma regra objetiva: escolher o fator para que a PP fique **acima do percentil 95 do IBGE** dos anos recentes (os
melhores produtores com a tecnologia atual). A proposta abaixo calcula esse fator; a decisão final é tomada com o
orientador e precisa ser defendida no artigo.

Como a calibração do ky usa produtividades **relativas** (PA/PP), o fator não muda o ky. Ele muda o tamanho do
yield gap em kg/ha (Etapa 7).

In [ ]:
recentes = cmp[cmp["ano"] >= 2019]
fator_proposto = np.quantile(recentes["PR_sobre_PP"], 0.95)
fator_proposto = max(1.0, round(float(fator_proposto), 2))
print(f"Fator proposto: {fator_proposto}  ->  Cc equivalente = {0.40 * fator_proposto:.2f}")

In [ ]:
# >>> DECISÃO (preencher depois de conversar com o orientador) <<<
FATOR_PP = fator_proposto          # ou um valor justificado, por exemplo 1.25 (Cc = 0,50)

with open(SAIDA / "parametros.json", "w") as f:
    json.dump({"FATOR_PP": FATOR_PP}, f, indent=2)
print("Gravado:", SAIDA / "parametros.json", {"FATOR_PP": FATOR_PP})

### O que escrever

No Material e Métodos, como a PP foi verificada e, se for o caso, ajustada (com a justificativa do Cc). Nos
Resultados, a figura `fig_pp_vs_ibge.png` e a tabela da comparação com o IDR e o GYGA.